
# TF‑IDF + XGBoost on SageMaker (with gzipped CSVs from S3)

End-to-end: read gzipped CSVs from S3, clean text, TF‑IDF features, train with AWS XGBoost, and (optionally) deploy an **inference pipeline** that accepts raw text.


In [ ]:

# %pip install --upgrade sagemaker scipy scikit-learn joblib
import os, boto3, joblib, re, json, gzip
import pandas as pd
import numpy as np
from io import BytesIO
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import dump_svmlight_file

sess   = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "tfidf-xgb-pipeline"
role   = sagemaker.get_execution_role()
region = sess.boto_region_name
print("Bucket:", bucket, "Region:", region)


In [ ]:

# === EDIT THESE ===
INPUT_BUCKET = "<your-source-bucket>"
INPUT_PREFIX = "raw-text-data/"
TEXT_COL     = "text"
LABEL_COL    = "label"
IS_MULTICLASS = False
NUM_CLASS     = 3

# TF-IDF params
NGRAM_RANGE   = (1, 2)
MIN_DF        = 2
MAX_DF        = 0.9
SUBLINEAR_TF  = True
NORM          = "l2"


In [ ]:

# List and load gzipped CSVs
s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=INPUT_BUCKET, Prefix=INPUT_PREFIX)
if "Contents" not in resp:
    raise ValueError(f"No objects under s3://{INPUT_BUCKET}/{INPUT_PREFIX}")
gz_files = [o["Key"] for o in resp["Contents"] if o["Key"].endswith(".csv.gz")]
print("Found gz files:", gz_files[:10])

frames = []
for key in gz_files:
    obj = s3.get_object(Bucket=INPUT_BUCKET, Key=key)
    with gzip.GzipFile(fileobj=BytesIO(obj["Body"].read())) as gz:
        frames.append(pd.read_csv(gz))

df = pd.concat(frames, ignore_index=True)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
print(df.shape)
df.head()


In [ ]:

# Cleaning
import re
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"http\S+", " ", s)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df[TEXT_COL].map(clean_text)

if df[LABEL_COL].dtype == object:
    classes = sorted(df[LABEL_COL].unique())
    cls2id = {c:i for i,c in enumerate(classes)}
    df["label_id"] = df[LABEL_COL].map(cls2id)
    print("Label mapping:", cls2id)
    y_col = "label_id"
else:
    y_col = LABEL_COL

df[[TEXT_COL, "text_clean", LABEL_COL]].head()


In [ ]:

# TF-IDF
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    sublinear_tf=SUBLINEAR_TF,
    norm=NORM,
    dtype=np.float32
)
X = tfidf.fit_transform(df["text_clean"])
y = df[y_col].astype(int).values
print("TF-IDF shape:", X.shape)

os.makedirs("data", exist_ok=True)
joblib.dump(tfidf, "data/vectorizer.pkl")


In [ ]:

# Split, save LIBSVM, upload
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dump_svmlight_file(Xtr, ytr, "data/train.libsvm", zero_based=True)
dump_svmlight_file(Xval, yval, "data/validation.libsvm", zero_based=True)

s3_train = sess.upload_data("data/train.libsvm", bucket=bucket, key_prefix=prefix)
s3_val   = sess.upload_data("data/validation.libsvm", bucket=bucket, key_prefix=prefix)
print("S3 train:", s3_train)
print("S3 val  :", s3_val)


In [ ]:

# Train XGBoost
xgb_image = image_uris.retrieve(framework="xgboost", region=region, version="1.7-1")
est = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=sess
)

if IS_MULTICLASS:
    est.set_hyperparameters(
        objective="multi:softprob",
        num_class=int(NUM_CLASS),
        eval_metric="mlogloss",
        num_round=800,
        max_depth=6,
        eta=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        tree_method="hist"
    )
else:
    est.set_hyperparameters(
        objective="binary:logistic",
        eval_metric="auc",
        num_round=800,
        max_depth=6,
        eta=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        tree_method="hist"
    )

est.fit({"train": s3_train, "validation": s3_val})


In [ ]:

# Option B: Inference pipeline artifacts
preprocess_script_path = "inference_artifacts/inference_preprocess.py"
os.makedirs("inference_artifacts", exist_ok=True)
with open(preprocess_script_path, "w") as f:
    f.write("""import json, joblib, numpy as np

def model_fn(model_dir):
    return joblib.load(f"{model_dir}/vectorizer.pkl")

def input_fn(request_body, content_type):
    if "application/json" in content_type or "json" in content_type:
        payload = json.loads(request_body)
        if isinstance(payload, dict) and "instances" in payload:
            return [str(t) for t in payload["instances"]]
        if isinstance(payload, list):
            return [str(t) for t in payload]
    raise ValueError("Unsupported content_type or payload format.")

def predict_fn(texts, vectorizer):
    X = vectorizer.transform([str(t) for t in texts])
    return X.astype(np.float32).toarray()

def output_fn(preds, accept):
    return json.dumps({"instances": preds.tolist()}), "application/json"
""")
print("Wrote", preprocess_script_path)


In [ ]:

# Upload artifacts and build pipeline
vec_s3 = sess.upload_data("data/vectorizer.pkl", bucket=bucket, key_prefix=f"{prefix}/pipeline-artifacts")
script_s3 = sess.upload_data("inference_artifacts/inference_preprocess.py", bucket=bucket, key_prefix=f"{prefix}/pipeline-artifacts")
print("Uploaded:", vec_s3)
print("Uploaded:", script_s3)

from sagemaker.sklearn.model import SKLearnModel
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.pipeline import PipelineModel

sklearn_model = SKLearnModel(
    framework_version="1.3-1",
    role=role,
    entry_point="inference_preprocess.py",
    model_data=vec_s3,
    sagemaker_session=sess,
)
xgb_model = XGBoostModel(
    model_data=est.model_data,
    role=role,
    framework_version="1.7-1",
    sagemaker_session=sess,
)
pipe_model = PipelineModel(
    name="tfidf-xgb-pipeline",
    role=role,
    models=[sklearn_model, xgb_model],
    sagemaker_session=sess
)
predictor = pipe_model.deploy(initial_instance_count=1, instance_type="ml.m5.large")
print("Endpoint:", predictor.endpoint_name)


In [ ]:

# Test endpoint
example_payload = {"instances": ["maven loves lemonade market", "lemon tea"]}
pred = predictor.predict(example_payload)
print(pred)

# Cleanup when finished:
# sagemaker.Session().delete_endpoint(predictor.endpoint_name)
